In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nikhilsinghbhadoriya/round2/search_results_3fold.csv
/kaggle/input/datasets/nikhilsinghbhadoriya/round2/notebookf8d43ed800.ipynb
/kaggle/input/datasets/nikhilsinghbhadoriya/round2/__notebook_source__.ipynb
/kaggle/input/datasets/nikhilsinghbhadoriya/round2/final_best_model_3fold.pt
/kaggle/input/datasets/nikhilsinghbhadoriya/round2/submission.csv
/kaggle/input/datasets/nikhilsinghbhadoriya/round2/test_features.csv


In [2]:
# ============================================================
# ROUND 2 INFERENCE ONLY
# Load saved final model -> predict test.csv -> save submission
# ============================================================

import os
import math
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================
@dataclass
class CFG:
    TEST_CSV: str = "/kaggle/input/datasets/nikhilsinghbhadoriya/round2/test_features.csv"
    MODEL_PATH: str = "/kaggle/input/datasets/nikhilsinghbhadoriya/round2/final_best_model_3fold.pt"

    OUTPUT_DIR: str = "/kaggle/working/"
    SUBMISSION_NAME: str = "submission_Test"

    seq_len: int = 100
    batch_size: int = 128
    num_workers: int = 2
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

cfg = CFG()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)


# ============================================================
# ROBUST COLUMN CLEANING
# ============================================================
def clean_column_name(col: str) -> str:
    col = str(col)
    col = col.replace("\ufeff", "")
    col = col.replace("\u200b", "")
    col = col.strip()

    if col.endswith("Sample_ID"):
        return "Sample_ID"
    if col.endswith("Time_ms"):
        return "Time_ms"
    if col.endswith("Is_Context"):
        return "Is_Context"
    if col.endswith("Value"):
        return "Value"
    return col


# ============================================================
# LOAD CSV
# ============================================================
def load_signal_csv(csv_path: str, seq_len: int = 100):
    print(f"Loading CSV: {csv_path}")
    df = pd.read_csv(csv_path)

    original_cols = list(df.columns)
    df.columns = [clean_column_name(c) for c in df.columns]
    cleaned_cols = list(df.columns)

    print("Original columns:", original_cols)
    print("Cleaned columns :", cleaned_cols)

    required = {"Sample_ID", "Time_ms", "Is_Context", "Value"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns after cleaning: {missing}")

    df = df[["Sample_ID", "Time_ms", "Is_Context", "Value"]].copy()

    df["Sample_ID"] = pd.to_numeric(df["Sample_ID"], errors="coerce")
    df["Time_ms"] = pd.to_numeric(df["Time_ms"], errors="coerce")
    df["Is_Context"] = pd.to_numeric(df["Is_Context"], errors="coerce")
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

    if df[["Sample_ID", "Time_ms", "Is_Context", "Value"]].isnull().any().any():
        raise ValueError("Found NaNs after numeric conversion. Please check CSV contents.")

    df = df.sort_values(["Sample_ID", "Time_ms"]).reset_index(drop=True)

    sample_ids_all = df["Sample_ID"].to_numpy(dtype=np.int64)
    time_ms_all = df["Time_ms"].to_numpy(dtype=np.float32)
    context_all = df["Is_Context"].to_numpy(dtype=np.float32)
    value_all = df["Value"].to_numpy(dtype=np.float32)

    uniq_ids, counts = np.unique(sample_ids_all, return_counts=True)
    if not np.all(counts == seq_len):
        raise ValueError(
            f"Every Sample_ID must have exactly {seq_len} rows. "
            f"Found min={counts.min()}, max={counts.max()}"
        )

    n_samples = len(uniq_ids)
    times = time_ms_all.reshape(n_samples, seq_len)
    context_mask = context_all.reshape(n_samples, seq_len)
    values = value_all.reshape(n_samples, seq_len)

    print(f"Loaded {n_samples:,} samples")
    print(f"Observed/context fraction: {context_mask.mean():.4f}")
    return uniq_ids, times, context_mask, values


# ============================================================
# NORMALIZATION
# ============================================================
def normalize_time_grid(times: np.ndarray) -> np.ndarray:
    tmin = times.min(axis=1, keepdims=True)
    tmax = times.max(axis=1, keepdims=True)
    return (times - tmin) / np.maximum(tmax - tmin, 1e-6)


def apply_value_normalization(values: np.ndarray, mean: float, std: float):
    return (values - mean) / std


# ============================================================
# DATASET
# ============================================================
class InferenceDataset(Dataset):
    def __init__(self, sample_ids, times, context_mask, values, mean, std):
        self.sample_ids = sample_ids
        self.raw_times = times.astype(np.float32)
        self.x = normalize_time_grid(times).astype(np.float32)
        self.context_mask = context_mask.astype(np.float32)
        self.values = apply_value_normalization(values.astype(np.float32), mean, std)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        x = self.x[idx]
        c = self.context_mask[idx]
        y = self.values[idx]

        observed_input_values = y * c
        target_mask = (1.0 - c).astype(np.float32)

        return {
            "sample_id": torch.tensor(self.sample_ids[idx], dtype=torch.long),
            "time_ms": torch.tensor(self.raw_times[idx], dtype=torch.float32),
            "x": torch.tensor(x, dtype=torch.float32),
            "visible_mask": torch.tensor(c, dtype=torch.float32),
            "target_mask": torch.tensor(target_mask, dtype=torch.float32),
            "observed_input_values": torch.tensor(observed_input_values, dtype=torch.float32),
        }


# ============================================================
# MODEL
# ============================================================
def sinusoidal_features(x, num_bands=16):
    device = x.device
    freqs = 2.0 ** torch.arange(num_bands, device=device, dtype=torch.float32) * math.pi
    x_exp = x.unsqueeze(-1)
    angles = x_exp * freqs.view(1, 1, -1)
    return torch.cat([x_exp, torch.sin(angles), torch.cos(angles)], dim=-1)


class ConvResidualBlock(nn.Module):
    def __init__(self, channels, dilation=1, dropout=0.1):
        super().__init__()
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        x = self.conv1(F.gelu(self.bn1(x)))
        x = self.dropout(x)
        x = self.conv2(F.gelu(self.bn2(x)))
        x = self.dropout(x)
        return x + residual


class HybridConvEncoder(nn.Module):
    def __init__(self, input_dim, d_model, num_blocks=4, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Conv1d(input_dim, d_model, kernel_size=1)
        self.blocks = nn.ModuleList([
            ConvResidualBlock(d_model, dilation=(2 ** i), dropout=dropout)
            for i in range(num_blocks)
        ])
        self.out_norm = nn.LayerNorm(d_model)

    def forward(self, feats):
        x = feats.transpose(1, 2)
        x = self.in_proj(x)
        for block in self.blocks:
            x = block(x)
        x = x.transpose(1, 2)
        return self.out_norm(x)


class CrossAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_mult=4, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * ff_mult),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * ff_mult, d_model),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, q, memory, memory_key_padding_mask=None):
        q2, _ = self.self_attn(q, q, q, need_weights=False)
        q = self.norm1(q + q2)

        q2, _ = self.cross_attn(
            q, memory, memory,
            key_padding_mask=memory_key_padding_mask,
            need_weights=False
        )
        q = self.norm2(q + q2)

        q2 = self.ff(q)
        q = self.norm3(q + q2)
        return q


class CrossAttentionDecoder(nn.Module):
    def __init__(self, time_dim, d_model, num_heads, num_layers, ff_mult=4, dropout=0.1):
        super().__init__()
        self.query_proj = nn.Sequential(
            nn.Linear(time_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.layers = nn.ModuleList([
            CrossAttentionBlock(d_model, num_heads, ff_mult=ff_mult, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.out_norm = nn.LayerNorm(d_model)

    def forward(self, query_time_features, memory, memory_key_padding_mask=None):
        q = self.query_proj(query_time_features)
        for layer in self.layers:
            q = layer(q, memory, memory_key_padding_mask=memory_key_padding_mask)
        return self.out_norm(q)


class HybridInpaintingModel(nn.Module):
    def __init__(self, hp):
        super().__init__()

        if hp["d_model"] % hp["num_heads"] != 0:
            raise ValueError(
                f"Invalid config: d_model={hp['d_model']} must be divisible by num_heads={hp['num_heads']}"
            )

        self.num_bands = hp["num_bands"]
        time_dim = 1 + 2 * self.num_bands
        input_dim = 2 + time_dim

        self.encoder = HybridConvEncoder(
            input_dim=input_dim,
            d_model=hp["d_model"],
            num_blocks=hp["num_conv_blocks"],
            dropout=hp["dropout"],
        )

        self.decoder = CrossAttentionDecoder(
            time_dim=time_dim,
            d_model=hp["d_model"],
            num_heads=hp["num_heads"],
            num_layers=hp["num_decoder_layers"],
            ff_mult=hp["ff_mult"],
            dropout=hp["dropout"],
        )

        self.head = nn.Sequential(
            nn.Linear(hp["d_model"], hp["d_model"]),
            nn.GELU(),
            nn.Dropout(hp["dropout"]),
            nn.Linear(hp["d_model"], 1),
        )

    def forward(self, x, observed_input_values, visible_mask):
        time_feats = sinusoidal_features(x, num_bands=self.num_bands)
        enc_feats = torch.cat([
            observed_input_values.unsqueeze(-1),
            visible_mask.unsqueeze(-1),
            time_feats
        ], dim=-1)

        memory = self.encoder(enc_feats)
        memory_key_padding_mask = (visible_mask < 0.5)

        decoded = self.decoder(
            query_time_features=time_feats,
            memory=memory,
            memory_key_padding_mask=memory_key_padding_mask
        )

        pred = self.head(decoded).squeeze(-1)
        return pred


# ============================================================
# PREDICT
# ============================================================
@torch.no_grad()
def predict_dataset(model, loader, device, mean, std):
    model.eval()

    all_sample_ids = []
    all_time_ms = []
    all_target_masks = []
    all_preds = []

    for batch in loader:
        sample_id = batch["sample_id"].cpu().numpy()       # [B]
        time_ms = batch["time_ms"].cpu().numpy()           # [B, L]
        x = batch["x"].to(device)
        visible_mask = batch["visible_mask"].to(device)
        observed_input_values = batch["observed_input_values"].to(device)
        target_mask = batch["target_mask"].cpu().numpy()

        pred = model(x, observed_input_values, visible_mask)
        pred = pred.cpu().numpy() * std + mean

        all_sample_ids.append(sample_id)
        all_time_ms.append(time_ms)
        all_target_masks.append(target_mask)
        all_preds.append(pred)

    all_sample_ids = np.concatenate(all_sample_ids, axis=0)      # [N]
    all_time_ms = np.concatenate(all_time_ms, axis=0)            # [N, L]
    all_target_masks = np.concatenate(all_target_masks, axis=0)  # [N, L]
    all_preds = np.concatenate(all_preds, axis=0)                # [N, L]

    return all_sample_ids, all_time_ms, all_target_masks, all_preds


# ============================================================
# MAIN
# ============================================================
print("Device:", cfg.device)

# 1) Load saved model
ckpt = torch.load(cfg.MODEL_PATH, map_location=cfg.device)
hp = ckpt["hp"]
value_mean = float(ckpt["value_mean"])
value_std = float(ckpt["value_std"])

print("Loaded model config:")
print(hp)
print(f"value_mean={value_mean:.6f}, value_std={value_std:.6f}")

model = HybridInpaintingModel(hp).to(cfg.device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# 2) Load Round 2 test.csv
test_sample_ids, test_times, test_context_mask, test_values = load_signal_csv(
    cfg.TEST_CSV,
    seq_len=cfg.seq_len
)

# 3) Build inference dataset
test_ds = InferenceDataset(
    sample_ids=test_sample_ids,
    times=test_times,
    context_mask=test_context_mask,
    values=test_values,
    mean=value_mean,
    std=value_std,
)

test_loader = DataLoader(
    test_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
    drop_last=False,
)

# 4) Predict
pred_sample_ids, pred_time_ms, pred_target_masks, pred_values = predict_dataset(
    model, test_loader, cfg.device, value_mean, value_std
)

print("Shapes before flatten:")
print("pred_sample_ids   :", pred_sample_ids.shape)
print("pred_time_ms      :", pred_time_ms.shape)
print("pred_target_masks :", pred_target_masks.shape)
print("pred_values       :", pred_values.shape)

# 5) Flatten correctly
n_samples, seq_len = pred_values.shape
flat_sample_ids = np.repeat(pred_sample_ids, seq_len)
flat_time_ms = pred_time_ms.reshape(-1)
flat_target_masks = pred_target_masks.reshape(-1)
flat_pred_values = pred_values.reshape(-1)

assert len(flat_sample_ids) == len(flat_time_ms) == len(flat_target_masks) == len(flat_pred_values)

# 6) Build submission
submission = pd.DataFrame({
    "Sample_ID": flat_sample_ids.astype(np.int64),
    "Time_ms": flat_time_ms.astype(np.int64),
    "Predicted_Value": flat_pred_values.astype(np.float32),
})

submission = submission[flat_target_masks > 0.5].copy().reset_index(drop=True)

submission_path = os.path.join(cfg.OUTPUT_DIR, cfg.SUBMISSION_NAME)
submission.to_csv(submission_path, index=False)

print(f"\nSaved submission to: {submission_path}")
print("Submission shape:", submission.shape)
print(submission.head(10))

Device: cuda
Loaded model config:
{'d_model': 192, 'num_conv_blocks': 4, 'num_heads': 4, 'num_decoder_layers': 4, 'ff_mult': 4, 'num_bands': 16, 'dropout': 0.15, 'lr': 0.00015, 'weight_decay': 0.0003, 'batch_size': 128, 'epochs': 80, 'early_stop_patience': 7, 'grad_clip': 1.0, 'mean_val_masked_mse': 0.029541633460919065, 'std_val_masked_mse': 0.0006088687628778629, 'mean_best_epoch': 54}
value_mean=0.500069, value_std=0.248696
Loading CSV: /kaggle/input/datasets/nikhilsinghbhadoriya/round2/test_features.csv
Original columns: ['Sample_ID', 'Time_ms', 'Is_Context', 'Value']
Cleaned columns : ['Sample_ID', 'Time_ms', 'Is_Context', 'Value']
Loaded 10,000 samples
Observed/context fraction: 0.2000
Shapes before flatten:
pred_sample_ids   : (10000,)
pred_time_ms      : (10000, 100)
pred_target_masks : (10000, 100)
pred_values       : (10000, 100)

Saved submission to: /kaggle/working/submission_Test
Submission shape: (800000, 3)
   Sample_ID  Time_ms  Predicted_Value
0      90000        1    